# Assignment 2 — Part 3

**CNN calculations and PyTorch verification**  
This notebook answers all six theory questions from the provided Assignment 2 instructions. Each answer includes a mathematical derivation (or conceptual explanation), followed by a short Python verification. **Part 1 (architecture) and Part 2 (CIFAR10 classifier/API) are separate deliverables.**

For a convolution or pooling operation (assuming dilation = 1), the spatial output size along one dimension is

$$O=\left\lfloor\frac{N+2P-K}{S}\right\rfloor+1,$$

where $N$ = input size, $K$ = kernel size, $P$ = padding on each side, and $S$ = stride. For 2D inputs, apply the formula to height and width separately. Output channel count is the **number of filters** in convolution, whereas pooling preserves the channel count.

In [2]:
import sys
print(sys.executable)

/opt/anaconda3/envs/aml1_env/bin/python


In [4]:
%pip install torch torchvision

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 MB 10.2 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 13.2 MB/s  0:00:00
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 8.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 28.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [torchvision] [torchvision]
Note: you may need to restart the kernel to use updated packages.


In [5]:
import math
import torch
import torch.nn as nn

def output_size(n, kernel, stride=1, padding=0):
    return math.floor((n + 2 * padding - kernel) / stride) + 1

print('PyTorch version:', torch.__version__)
print('Formula helper ready.')

PyTorch version: 2.8.0
Formula helper ready.


### Question 1 

**Question:** Given an input image of size $32\times32\times3$ and a convolutional layer with **8 filters** of size $5\times5$, stride $1$, and no padding, what is the output size?

**Given:** $N=32$, $K=5$, $S=1$, $P=0$.

$$H_{out}=W_{out}=\left\lfloor\frac{32+2(0)-5}{1}\right\rfloor+1=27+1=28.$$

Each of the 8 filters produces one output feature map. Thus, the **output shape is $28\times28\times8$** (height × width × channels). In PyTorch's batch-first convention it is $(1,8,28,28)$ for one image.

In [6]:
size = output_size(32, kernel=5, stride=1, padding=0)
x = torch.randn(1, 3, 32, 32)
y = nn.Conv2d(in_channels=3, out_channels=8, kernel_size=5, stride=1, padding=0)(x)
assert y.shape == (1, 8, 28, 28)
print('Calculated H x W x C:', (size, size, 8))
print('PyTorch (N, C, H, W):', tuple(y.shape))

Calculated H x W x C: (28, 28, 8)
PyTorch (N, C, H, W): (1, 8, 28, 28)


## Question 2

**Question:** How does the output size change if the padding in Question 1 is changed to **“same”**?

With stride $1$, “same” padding preserves the spatial height and width. For an odd $5\times5$ kernel, use $P=(5-1)/2=2$ on each side:

$$H_{out}=W_{out}=\left\lfloor\frac{32+2(2)-5}{1}\right\rfloor+1=32.$$

The 8 filters still give 8 output channels. The **new output is $32\times32\times8$**, compared with $28\times28\times8$ in Question 1. Each spatial dimension increases by 4 relative to the no-padding result.

In [7]:
size = output_size(32, kernel=5, stride=1, padding=2)
y = nn.Conv2d(3, 8, kernel_size=5, stride=1, padding='same')(torch.randn(1, 3, 32, 32))
assert y.shape == (1, 8, 32, 32)
print('Calculated H x W x C:', (size, size, 8))
print('PyTorch (N, C, H, W):', tuple(y.shape))

Calculated H x W x C: (32, 32, 8)
PyTorch (N, C, H, W): (1, 8, 32, 32)


## Question 3

**Question:** If you apply a $3\times3$ filter with stride $2$ and no padding to a $64\times64$ input, what is the output **spatial** size?

**Given:** $N=64$, $K=3$, $S=2$, $P=0$.

$$H_{out}=W_{out}=\left\lfloor\frac{64-3}{2}\right\rfloor+1=\lfloor30.5\rfloor+1=31.$$

Therefore, the **spatial output size is $31\times31$**. The question does not specify a number of filters, so we cannot determine the output channel count.

In [8]:
size = output_size(64, kernel=3, stride=2, padding=0)
y = nn.Conv2d(1, 1, kernel_size=3, stride=2, padding=0)(torch.randn(1, 1, 64, 64))
assert y.shape[-2:] == (31, 31)
print('Calculated spatial size:', (size, size))
print('PyTorch spatial size:', tuple(y.shape[-2:]))

Calculated spatial size: (31, 31)
PyTorch spatial size: (31, 31)


## Question 4 

**Question:** You apply a max-pooling layer of size $2\times2$ with stride $2$ to a $16\times16$ feature map. What is the output size?

**Given:** $N=16$, $K=2$, $S=2$, $P=0$ (default).

$$H_{out}=W_{out}=\left\lfloor\frac{16-2}{2}\right\rfloor+1=7+1=8.$$

Therefore, the **spatial output size is $8\times8$**. Max pooling reduces the spatial resolution but does not change the number of channels.

In [9]:
size = output_size(16, kernel=2, stride=2, padding=0)
y = nn.MaxPool2d(kernel_size=2, stride=2)(torch.randn(1, 1, 16, 16))
assert y.shape[-2:] == (8, 8)
print('Calculated spatial size:', (size, size))
print('PyTorch spatial size:', tuple(y.shape[-2:]))

Calculated spatial size: (8, 8)
PyTorch spatial size: (8, 8)


## Question 5 

**Question:** An image of shape $128\times128$ is passed through two successive convolutional layers. Each uses a $3\times3$ kernel, stride $1$, and “same” padding. What is the output shape?

A $3\times3$ kernel with stride 1 uses padding $P=1$ on each side to preserve spatial size.

**Layer 1:**
$$H_1=W_1=\left\lfloor\frac{128+2(1)-3}{1}\right\rfloor+1=128.$$

**Layer 2:**
$$H_2=W_2=\left\lfloor\frac{128+2(1)-3}{1}\right\rfloor+1=128.$$

Therefore, the **final spatial output shape is $128\times128$**. The full channel dimension cannot be specified because the number of filters in the second convolution is not given.

In [10]:
first = output_size(128, kernel=3, stride=1, padding=1)
second = output_size(first, kernel=3, stride=1, padding=1)
network = nn.Sequential(nn.Conv2d(1, 4, 3, stride=1, padding='same'),
                        nn.Conv2d(4, 6, 3, stride=1, padding='same'))
y = network(torch.randn(1, 1, 128, 128))
assert y.shape[-2:] == (128, 128)
print('After layer 1 (spatial):', (first, first))
print('After layer 2 (spatial):', (second, second))
print('PyTorch spatial size:', tuple(y.shape[-2:]))
print('Note: 4 and 6 filters are illustrative, not specified in the question.')

After layer 1 (spatial): (128, 128)
After layer 2 (spatial): (128, 128)
PyTorch spatial size: (128, 128)
Note: 4 and 6 filters are illustrative, not specified in the question.


## Question 6 

**Question:** In the class examples, before starting the training loop we ran `model.train()`. What happens if you remove that line?

`model.train()` sets the model and its submodules to **training mode**. This matters for layers that behave differently during training and evaluation:

- **Dropout:** randomly drops activations in training mode; it is disabled during evaluation.
- **BatchNorm:** typically uses the current mini-batch statistics and updates running statistics during training; evaluation generally uses stored running statistics.

**Important nuance:** A newly created PyTorch model starts in training mode **by default**. Therefore, if no previous call changed its mode, deleting `model.train()` may make **no immediate difference**. However, if `model.eval()` was called earlier (for example, to evaluate after an epoch), and you do not switch back to training mode, Dropout and BatchNorm continue behaving as in evaluation mode. That changes training behavior and can affect results.

`model.train()` **does not** itself run gradient descent or turn gradients on. Those are controlled by operations such as `loss.backward()`, `optimizer.step()`, and gradient-mode controls such as `torch.no_grad()`.

**Answer:** Removing the line is not always an error for a fresh model, but explicitly calling it before training is best practice because it ensures training-specific layers are in the correct mode.

In [11]:
# Verify that a fresh model defaults to training mode, and demonstrate Dropout.
torch.manual_seed(42)
model = nn.Sequential(nn.Dropout(p=0.5))
x = torch.ones(1000)
print('New model training mode:', model.training)
model.train()
y_train = model(x)
print('Training mode: number of zeros from Dropout:', int((y_train == 0).sum()))
model.eval()
y_eval = model(x)
print('Evaluation mode:', model.training)
print('Evaluation mode: unchanged input:', torch.equal(y_eval, x))
model.train()
print('After calling model.train() again:', model.training)

New model training mode: True
Training mode: number of zeros from Dropout: 493
Evaluation mode: False
Evaluation mode: unchanged input: True
After calling model.train() again: True
